# Lab 1 - Exercise 3.1: Improving Fine-tuning Performance

Questo notebook documenta il passaggio dalla baseline head-only dell'Exercise 1.3 a un fine-tuning piu' efficace. L'obiettivo non e' provare molte idee in modo casuale, ma confrontare poche varianti motivate e discutere cosa ci dicono i risultati.

I risultati riportati qui sono quelli degli artifact della pipeline nuova in `artifacts/runs/`. Le run storiche sono conservate in `Esperimenti_di_prova.ipynb`, ma non vengono usate per trarre le conclusioni finali.


---
### Exercise 3.1 (Easy): Improving Fine-tuning Performance

In this exercise you are asked to iterate on the fine-tuning experiment performed in Exercise 1.3 in order to squeeze the best performance possible out of the model.

What can we do:

- Use a more powerful model?
- More aggressive data augmentation?
- Selective layer training?
- Something else?

L'idea scelta qui e' **selective layer training**: invece di allenare solo la `fc`, sblocchiamo anche `layer4`, cioe' l'ultimo blocco residuo della ResNet-18. E' un compromesso semplice: adattiamo le feature piu' alte al dominio GTSRB senza riaddestrare tutta la rete.

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dla_lab1.config import experiment_config, load_config
from dla_lab1.experiments import batch_size_for, run_finetuning
from dla_lab1.evaluate import classification_metrics, history_to_frame, predict
from dla_lab1.models import build_classifier, count_parameters

config = load_config(ROOT / "config" / "config.yaml")

# Non rilanciamo training lunghi automaticamente.
RUN_TRAINING = True

print(f"Project root: {ROOT}")


Project root: c:\Users\checc\OneDrive\Desktop\DLA\DLA_Lab\DLA_1


## Punto di partenza: baseline Exercise 1.3

La baseline head-only e' il riferimento metodologico: ResNet-18 pre-addestrata, backbone congelato e solo classificatore finale addestrabile.

In questo notebook non copiamo valori numerici dal vecchio lavoro. Il confronto deve essere fatto con risultati prodotti dalla pipeline corrente o con artifact generati localmente.


In [2]:
baseline_context = pd.DataFrame([
    ["Exercise 1.2", "ResNet-18 feature extractor + SVM", "baseline stabile senza fine-tuning"],
    ["Exercise 1.3", "ResNet-18 frozen backbone + linear fc", "fine-tuning head-only"],
], columns=["Reference", "Setup", "Role"])

baseline_context


,Reference,Setup,Role
0,Exercise 1.2,ResNet-18 feature extractor + SVM,baseline stabile senza fine-tuning
1,Exercise 1.3,ResNet-18 frozen backbone + linear fc,fine-tuning head-only


Nota: i risultati prodotti in notebook vecchi possono aiutare a ricordare quali idee erano state provate, ma non devono essere usati come risultati finali in questa versione. Qui manteniamo solo il metodo e rilanciamo le configurazioni rilevanti nella pipeline corrente.


## Esperimenti possibili per Exercise 3.1

Le idee da provare sono tre:

- sbloccare `layer4` mantenendo congelato il resto del backbone;
- aggiungere augmentation moderata;
- combinare `layer4` trainabile e augmentation.

La tabella seguente elenca le configurazioni disponibili, senza riportare risultati copiati dal vecchio notebook.


In [3]:
improvement_candidates = pd.DataFrame([
    ["ex3_1_layer4_unfrozen", "layer4 + fc", "none", "verifica dell'effetto dello sblocco di layer4"],
    ["ex3_1_head_only_aggressive_aug", "fc only", "aggressive", "controllo: augmentation senza fine-tuning del backbone"],
    ["ex3_1_layer4_aggressive_aug", "layer4 + fc", "aggressive", "augmentation forte con layer4 trainabile"],
    ["ex3_1_layer4_conservative_aug", "layer4 + fc", "conservative", "augmentation realistica con layer4 trainabile"],
    ["ex3_1_layer4_spatial_aug", "layer4 + fc", "spatial", "sole perturbazioni geometriche leggere"],
    ["ex3_1_layer4_no_aug_lr1e4_wd05", "layer4 + fc", "none", "LR piu basso e weight decay piu alto senza augmentation"],
    ["ex3_1_layer4_safe_aug_ls005_discriminative", "layer4 + fc", "safe", "augmentation sicura, label smoothing e LR discriminativi"],
    ["ex3_1_layer4_conservative_ls005", "layer4 + fc", "conservative", "label smoothing su augmentation conservativa"],
], columns=["Experiment", "Trainable layers", "Augmentation", "Purpose"])

improvement_candidates


,Experiment,Trainable layers,Augmentation,Purpose
0,ex3_1_layer4_unfrozen,layer4 + fc,none,verifica dell'effetto dello sblocco di layer4
1,ex3_1_head_only_aggressive_aug,fc only,aggressive,controllo: augmentation senza fine-tuning del ...
2,ex3_1_layer4_aggressive_aug,layer4 + fc,aggressive,augmentation forte con layer4 trainabile
3,ex3_1_layer4_conservative_aug,layer4 + fc,conservative,augmentation realistica con layer4 trainabile
4,ex3_1_layer4_spatial_aug,layer4 + fc,spatial,sole perturbazioni geometriche leggere
5,ex3_1_layer4_no_aug_lr1e4_wd05,layer4 + fc,none,LR piu basso e weight decay piu alto senza aug...
6,ex3_1_layer4_safe_aug_ls005_discriminative,layer4 + fc,safe,"augmentation sicura, label smoothing e LR disc..."
7,ex3_1_layer4_conservative_ls005,layer4 + fc,conservative,label smoothing su augmentation conservativa


## Strategia sperimentale

La baseline head-only allena solo la `fc` finale. Se le feature ImageNet non separano bene i segnali GTSRB, questa scelta e' troppo rigida. Per questo le varianti dell'Exercise 3.1 provano a sbloccare `layer4`, cioe' l'ultimo blocco convoluzionale della ResNet-18, lasciando congelati i blocchi precedenti.

Le augmentation vengono valutate con cautela. Per GTSRB alcune trasformazioni sono semanticamente rischiose: ad esempio il flip orizzontale puo' cambiare il significato di segnali direzionali. Per questo la variante aggressiva e' utile come esperimento, ma non e' automaticamente una buona scelta finale.


## Risultati della pipeline corrente

| Run | Parametri allenati | Augmentation | Best val acc | Ultima train acc | Ultima val acc | Gap train-val finale |
|---|---|---|---:|---:|---:|---:|
| `ex1_3_head_only_adam_ce` | `fc` | none | 0.4665 | 0.8272 | 0.4653 | 0.3619 |
| `ex3_1_head_only_aggressive_aug` | `fc` | aggressive | 0.4387 | 0.6678 | 0.4296 | 0.2383 |
| `ex3_1_layer4_aggressive_aug` | `layer4` + `fc` | aggressive | 0.7433 | 0.9927 | 0.7369 | 0.2557 |
| `ex3_1_layer4_conservative_aug` | `layer4` + `fc` | conservative | 0.7541 | 0.9979 | 0.7512 | 0.2467 |
| `ex3_1_layer4_spatial_aug` | `layer4` + `fc` | spatial | 0.7089 | 0.9986 | 0.7089 | 0.2897 |
| `ex3_1_layer4_unfrozen` | `layer4` + `fc` | none | 0.7584 | 1.0000 | 0.7584 | 0.2416 |
| `ex3_1_layer4_no_aug_lr1e4_wd05` | `layer4` + `fc` | none | 0.6947 | 1.0000 | 0.6938 | 0.3062 |
| `ex3_1_layer4_safe_aug_ls005_discriminative` | `layer4` + `fc` | safe | 0.7306 | 0.9985 | 0.7282 | 0.2703 |
| `ex3_1_layer4_conservative_ls005` | `layer4` + `fc` | conservative | **0.7741** | 0.9998 | 0.7701 | 0.2297 |

La configurazione migliore e' `ex3_1_layer4_conservative_ls005`. Il risultato mostra che, in questo setup, il contributo piu' efficace non e' l'augmentation aggressiva ma una regolarizzazione leggera della loss (`label_smoothing=0.05`) insieme a trasformazioni realistiche per i segnali stradali. La variante `safe` con learning rate discriminativi e' piu' prudente ma non raggiunge la stessa accuratezza; il solo abbassamento del learning rate senza augmentation peggiora sensibilmente la validazione.


In [4]:
candidate_runs = [
    "ex3_1_layer4_unfrozen",
    "ex3_1_layer4_conservative_aug",
    "ex3_1_layer4_spatial_aug",
    "ex3_1_layer4_no_aug_lr1e4_wd05",
    "ex3_1_layer4_safe_aug_ls005_discriminative",
    "ex3_1_layer4_conservative_ls005",
]

run_rows = []
for run_name in candidate_runs:
    summary_path = ROOT / "artifacts" / "runs" / run_name / "summary.json"
    if summary_path.exists():
        summary = pd.read_json(summary_path, typ="series")
        run_rows.append([
            run_name,
            summary.get("best_epoch", None),
            summary.get("best_val_acc", None),
            summary_path,
        ])

if run_rows:
    current_ex3_results = pd.DataFrame(
        run_rows,
        columns=["Experiment", "Best Epoch", "Best Val Accuracy", "Summary Path"],
    ).sort_values("Best Val Accuracy", ascending=False)
    display(current_ex3_results)
else:
    print("Nessun summary.json trovato per le run candidate.")


,Experiment,Best Epoch,Best Val Accuracy,Summary Path
5,ex3_1_layer4_conservative_ls005,9.0,0.774055,c:\Users\checc\OneDrive\Desktop\DLA\DLA_Lab\DL...
0,ex3_1_layer4_unfrozen,15.0,0.758419,c:\Users\checc\OneDrive\Desktop\DLA\DLA_Lab\DL...
1,ex3_1_layer4_conservative_aug,9.0,0.754124,c:\Users\checc\OneDrive\Desktop\DLA\DLA_Lab\DL...
4,ex3_1_layer4_safe_aug_ls005_discriminative,15.0,0.730584,c:\Users\checc\OneDrive\Desktop\DLA\DLA_Lab\DL...
2,ex3_1_layer4_spatial_aug,15.0,0.708935,c:\Users\checc\OneDrive\Desktop\DLA\DLA_Lab\DL...
3,ex3_1_layer4_no_aug_lr1e4_wd05,13.0,0.694674,c:\Users\checc\OneDrive\Desktop\DLA\DLA_Lab\DL...


## Configurazioni nella nuova pipeline

Le configurazioni utili per Exercise 3.1 sono ora in `config/config.yaml`. Questo permette di rilanciare le prove senza riscrivere celle lunghe nel notebook.

In [5]:
exercise_31_experiments = [
    "ex3_1_layer4_unfrozen",
    "ex3_1_layer4_conservative_aug",
    "ex3_1_layer4_spatial_aug",
    "ex3_1_layer4_no_aug_lr1e4_wd05",
    "ex3_1_layer4_safe_aug_ls005_discriminative",
    "ex3_1_layer4_conservative_ls005",
]

config_rows = []
for name in exercise_31_experiments:
    exp = experiment_config(config, name)
    config_rows.append([
        name,
        exp["model"]["name"],
        exp["model"]["unfreeze_layer4"],
        exp["experiment"].get("augmentation", "none"),
        exp["training"]["optimizer"],
        exp["training"]["loss"],
        exp["training"].get("label_smoothing", 0.0),
        exp["training"]["learning_rate"],
        exp["training"].get("layer4_lr", None),
        exp["training"].get("fc_lr", None),
        exp["training"]["epochs"],
        batch_size_for(config, exp["experiment"]["batch_size_key"]),
    ])

pd.DataFrame(config_rows, columns=[
    "Experiment", "Model", "Unfreeze layer4", "Augmentation", "Optimizer",
    "Loss", "Label smoothing", "Learning rate", "Layer4 LR", "FC LR",
    "Epochs", "Batch size"
])


,Experiment,Model,Unfreeze layer4,Augmentation,Optimizer,Loss,Label smoothing,Learning rate,Layer4 LR,FC LR,Epochs,Batch size
0,ex3_1_layer4_unfrozen,resnet18,True,none,AdamW,CrossEntropy,0.00,0.0005,NaN,NaN,15,128
1,ex3_1_layer4_conservative_aug,resnet18,True,conservative,AdamW,CrossEntropy,0.00,0.0005,NaN,NaN,10,128
2,ex3_1_layer4_spatial_aug,resnet18,True,spatial,AdamW,CrossEntropy,0.00,0.0005,NaN,NaN,10,128
3,ex3_1_layer4_no_aug_lr1e4_wd05,resnet18,True,none,AdamW,CrossEntropy,0.00,0.0001,NaN,NaN,20,128
4,ex3_1_layer4_safe_aug_ls005_discriminative,resnet18,True,safe,AdamW,CrossEntropy,0.05,0.0005,0.0001,0.0005,20,128
5,ex3_1_layer4_conservative_ls005,resnet18,True,conservative,AdamW,CrossEntropy,0.05,0.0005,NaN,NaN,20,128


## Modello selezionato e overfitting

Il modello selezionato e' `ex3_1_layer4_conservative_ls005`, perche' ottiene la migliore validation accuracy tra le configurazioni confrontate: **0.7741** alla epoca 11. La scelta usa solo training e validation; il test set resta separato fino alla valutazione finale.

Il modello mostra ancora overfitting: la train accuracy finale e' **0.9998**, mentre la validation finale e' **0.7701**. Questo non indica data leakage. Indica che il blocco `layer4`, una volta sbloccato, ha capacita' sufficiente per adattarsi quasi perfettamente al training split. La regolarizzazione con `label_smoothing=0.05` migliora la validation e riduce leggermente il gap rispetto al precedente best, ma non annulla la differenza tra training e validation.

La causa principale e' la combinazione di tre fattori: GTSRB e' sbilanciato, molte immagini della stessa classe sono molto simili tra loro, e il fine-tuning di `layer4` aumenta molto la capacita' rispetto alla sola testa finale. Per questo motivo le trasformazioni aggressive non sono una soluzione appropriata: l'horizontal flip puo' cambiare il significato dei segnali direzionali, mentre variazioni cromatiche forti possono alterare colori semanticamente importanti.


In [6]:
SELECTED_EXPERIMENT = "ex3_1_layer4_conservative_ls005"
selected_cfg = experiment_config(config, SELECTED_EXPERIMENT)

model = build_classifier(
    model_name=selected_cfg["model"]["name"],
    num_classes=int(config["project"]["num_classes"]),
    weights=selected_cfg["model"].get("weights", "DEFAULT"),
    freeze_backbone=bool(selected_cfg["model"].get("freeze_backbone", True)),
    unfreeze_layer4=bool(selected_cfg["model"].get("unfreeze_layer4", False)),
)

parameter_summary = pd.DataFrame([count_parameters(model)])
parameter_summary["trainable_ratio"] = parameter_summary["trainable"] / parameter_summary["total"]
parameter_summary

,total,trainable,trainable_ratio
0,11198571,8415787,0.751505


Il numero di parametri trainabili e' molto piu' alto della baseline head-only, ma resta inferiore al fine-tuning completo. Questo e' il compromesso principale dell'esercizio: piu' adattamento al dataset senza aggiornare tutta la rete.

## Cella di riproducibilita' del miglior esperimento

La cella seguente e' disattivata di default. Serve solo se vuoi rieseguire la variante selezionata nella nuova pipeline.

In [7]:
if RUN_TRAINING:
    result = run_finetuning(config, SELECTED_EXPERIMENT, root=ROOT)
    history = history_to_frame(result["history"])
    display(history)
    print(f"Run salvata in: {result['artifacts']['output_dir']}")
else:
    print("Training non eseguito. Imposta RUN_TRAINING = True per rilanciare Exercise 3.1.")


  0%|          | 0/163 [00:09<?, ?it/s]

  0%|          | 0/46 [00:08<?, ?it/s]

Epoch 01 | lr=5.00e-04 | train_loss=0.9693 train_acc=0.8201 | val_loss=1.4982 val_acc=0.6765


  0%|          | 0/163 [00:00<?, ?it/s]

  0%|          | 0/46 [00:00<?, ?it/s]

...

  0%|          | 0/163 [00:00<?, ?it/s]

  0%|          | 0/46 [00:00<?, ?it/s]

Epoch 20 | lr=6.25e-05 | train_loss=0.3821 train_acc=0.9996 | val_loss=1.1819 val_acc=0.7687


,epoch,train_loss,train_acc,val_loss,val_acc,learning_rate
0,1,0.969325,0.820125,1.498222,0.676460,0.000500
1,2,0.529425,0.962488,1.372352,0.712199,0.000500
2,3,0.467647,0.979779,1.378344,0.704467,0.000500
3,4,0.438489,0.988088,1.334999,0.724055,0.000500
4,5,0.424933,0.989673,1.295144,0.739347,0.000500
5,6,0.399932,0.996830,1.231221,0.748797,0.000250
6,7,0.394043,0.998223,1.272751,0.744845,0.000250
7,8,0.391397,0.998415,1.262641,0.754467,0.000250
8,9,0.390107,0.998655,1.226413,0.758591,0.000250
9,10,0.390719,0.998511,1.201572,0.759622,0.000250


Run salvata in: c:\Users\checc\OneDrive\Desktop\DLA\DLA_Lab\DLA_1\artifacts\runs\ex3_1_layer4_conservative_ls005


## Valutazione finale sul test set del best model

Dopo la selezione su validation, il test set viene usato una sola volta per stimare la generalizzazione finale del modello scelto. Questa e' la procedura corretta: il test non partecipa alla scelta di augmentation, learning rate, label smoothing o checkpoint.

La cella seguente valuta direttamente sul test set il best model restituito dalla cella di training precedente (`result["model"]`). `train_model` ricarica gia' il checkpoint con la migliore validation accuracy, quindi questa valutazione usa il modello selezionato correttamente.

La valutazione finale eseguita sul test set produce **accuracy = 0.7949**, **macro F1 = 0.73** e **weighted F1 = 0.79**. Questo risultato non e' stato usato per modificare iperparametri o scegliere un nuovo checkpoint: serve solo come stima finale della generalizzazione.


In [8]:
RUN_FINAL_TEST_EVAL = True

if RUN_FINAL_TEST_EVAL:
    if "result" not in globals():
        raise RuntimeError("Esegui prima la cella di training che crea result = run_finetuning(...).")

    y_true, y_pred = predict(result["model"], result["loaders"]["test"], result["device"])
    final_metrics = classification_metrics(y_true.numpy(), y_pred.numpy())

    print(f"Test accuracy Exercise 3.1: {final_metrics['accuracy']:.4f}")
    print(final_metrics["classification_report"])
else:
    print(
        "Test finale non eseguito. "
        "Non usare il test per ulteriori scelte di iperparametri."
    )


Test accuracy Exercise 3.1: 0.8025
              precision    recall  f1-score   support

           0       0.72      0.35      0.47        60
           1       0.68      0.63      0.65       720
           2       0.62      0.73      0.67       750
           3       0.71      0.68      0.69       450
           4       0.74      0.83      0.78       660
           5       0.65      0.67      0.66       630
           6       0.90      0.93      0.91       150
           7       0.89      0.78      0.83       450
           8       0.72      0.74      0.73       450
           9       0.94      0.85      0.89       480
          10       0.91      0.99      0.95       660
          11       0.80      0.78      0.79       420
          12       1.00      0.99      0.99       690
          13       0.99      1.00      1.00       720
          14       0.94      0.97      0.96       270
          15       0.98      0.99      0.99       210
          16       1.00      0.92      0.96   

## Funzioni usate

Le funzioni principali sono concentrate nel package `src/dla_lab1`, in modo che il notebook resti leggibile e la pipeline sia riutilizzabile.

- `build_transforms` in `transforms.py` definisce preprocessing e augmentation (`none`, `aggressive`, `conservative`, `safe`, `spatial`).
- `build_classifier` in `models.py` costruisce ResNet-18 e applica la scelta di congelare o sbloccare `layer4`.
- `train_model` in `train.py` esegue il ciclo di training, salva il checkpoint migliore e applica early stopping.
- `trainable_parameter_groups` in `train.py` permette learning rate separati per `layer4` e `fc` quando richiesto dal config.
- `build_loss` in `losses.py` costruisce la loss e applica `label_smoothing` per la run finale.
- `run_finetuning` in `experiments.py` collega configurazione, dataloader, modello, training e salvataggio degli artifact.


## Conclusione Exercise 3.1

La progressione sperimentale e' coerente: la baseline head-only arriva a circa **46,6%** di validation accuracy, mentre il fine-tuning selettivo di `layer4` porta il modello oltre il **75%**. La migliore configurazione finale e' `ex3_1_layer4_conservative_ls005`, con **77,4%** di validation accuracy.

La scelta finale usa augmentation conservativa e `label_smoothing=0.05`. Questa combinazione e' adatta al dominio: introduce piccole variazioni realistiche di crop, rotazione, luminosita' e contrasto, senza trasformazioni semanticamente pericolose come horizontal flip o hue shift marcati. Il risultato e' superiore sia alla run senza augmentation sia alle varianti aggressive.

Il comportamento di overfitting resta visibile e viene riportato esplicitamente: la train accuracy quasi perfetta mostra che il modello si adatta molto bene al training set, mentre la validation misura la difficolta' di generalizzare su immagini non viste. La conclusione sperimentale e' quindi precisa: il fine-tuning di `layer4` e la regolarizzazione leggera migliorano la generalizzazione, ma il task rimane sensibile alla capacita' del modello e allo sbilanciamento delle classi.

Sul test set, valutato una sola volta dopo la selezione su validation, il checkpoint finale raggiunge **79,5%** di accuracy. La macro F1 pari a **0.73** e' piu' bassa della weighted F1 pari a **0.79**, quindi le classi rare restano piu' difficili rispetto alle classi frequenti. Questo e' coerente con l'analisi esplorativa iniziale, che aveva evidenziato uno sbilanciamento marcato nella distribuzione delle classi.

La procedura train/validation/test e' corretta. Train aggiorna i pesi, validation seleziona modello e iperparametri, test fornisce solo la stima finale dopo la scelta. In questo modo non c'e' leakage metodologico dal test set.
